In [7]:
# Лабораторная работа 4: Linear Discriminant Analysis в Python

#Задача: по химическим признакам вина `C1–C13` определить производителя `V1`, `V2`, `V3`.

#В ноутбуке реализованы два варианта:

#1. **LINEST-аналог**, как в Excel через линейную модель.
#2. **Solver-аналог**, где коэффициенты оптимизируются по отношению `Inter-group variance / Within-group variance`.

In [8]:
import pandas as pd
import numpy as np

file_path = "Chapter4-HW.xlsx"

raw = pd.read_excel(file_path, sheet_name="LDA")

# Берем только исходные данные: Vendor + C1:C13
data = raw.iloc[:, :14].copy()

# Обучающие строки: известные производители V1, V2, V3
train = data[data["Vendor"].isin(["V1", "V2", "V3"])].copy()

# Строки для классификации: Vendor = ?
unknown = data[data["Vendor"].eq("?")].copy()

features = [f"C{i}" for i in range(1, 14)]

train.head()

,Vendor,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13
0,V1,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,V1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,V1,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,V1,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,V1,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [9]:
# Кодируем классы как в Excel:
# V1 -> 0
# V2 -> 1
# V3 -> 2

class_to_num = {
    "V1": 0,
    "V2": 1,
    "V3": 2
}

num_to_class = {
    0: "V1",
    1: "V2",
    2: "V3"
}

X = train[features].astype(float).values
y = train["Vendor"].map(class_to_num).astype(float).values

# Добавляем столбец единиц для свободного коэффициента b
X_design = np.column_stack([np.ones(len(X)), X])

print("Размер обучающей выборки:", X.shape)
print("Количество объектов по классам:")
print(train["Vendor"].value_counts().sort_index())

Размер обучающей выборки: (178, 13)
Количество объектов по классам:
Vendor
V1    59
V2    71
V3    48
Name: count, dtype: int64


In [10]:
## 1. LINEST-аналог

#В Excel использовалась функция `ЛИНЕЙН`.

#В Python то же самое можно сделать методом наименьших квадратов:

#\[
#y = b + w_1x_1 + w_2x_2 + ... + w_{13}x_{13}
#\]


In [11]:
# Аналог Excel LINEST
linest_coefficients = np.linalg.lstsq(X_design, y, rcond=None)[0]

coef_names = ["b"] + [f"w{i}" for i in range(1, 14)]

linest_table = pd.DataFrame({
    "coefficient": coef_names,
    "value": linest_coefficients
})

linest_table

,coefficient,value
0,b,3.473285
1,w1,-0.117004
2,w2,0.030171
3,w3,-0.148552
4,w4,0.039854
5,w5,-0.000490
6,w6,0.144320
7,w7,-0.372391
8,w8,-0.303474
9,w9,0.039357


In [12]:
def calculate_predictions(X_design, coefficients):
    return X_design @ coefficients


def calculate_means_counts_cutoffs(predictions, y):
    result_df = pd.DataFrame({
        "y": y,
        "prediction": predictions
    })

    means = result_df.groupby("y")["prediction"].mean()
    counts = result_df.groupby("y")["prediction"].count()

    cutoff_01 = (means.loc[0] * counts.loc[0] + means.loc[1] * counts.loc[1]) / (counts.loc[0] + counts.loc[1])
    cutoff_12 = (means.loc[1] * counts.loc[1] + means.loc[2] * counts.loc[2]) / (counts.loc[1] + counts.loc[2])

    return means, counts, [cutoff_01, cutoff_12]


def classify_by_cutoffs(predictions, cutoffs):
    classes = []

    for value in predictions:
        if value < cutoffs[0]:
            classes.append("V1")
        elif value < cutoffs[1]:
            classes.append("V2")
        else:
            classes.append("V3")

    return np.array(classes)


def count_errors(real_classes, predicted_classes):
    return (real_classes.values != predicted_classes).sum()


def lda_metrics(coefficients, X_design, y):
    predictions = calculate_predictions(X_design, coefficients)
    means, counts, cutoffs = calculate_means_counts_cutoffs(predictions, y)

    inter_group = ((means - means.mean()) ** 2).sum()

    within_group = 0
    for class_number in [0, 1, 2]:
        class_predictions = predictions[y == class_number]
        within_group += ((class_predictions - means.loc[class_number]) ** 2).sum()

    ratio = inter_group / within_group

    return {
        "predictions": predictions,
        "means": means,
        "counts": counts,
        "cutoffs": cutoffs,
        "inter_group": inter_group,
        "within_group": within_group,
        "ratio": ratio
    }

In [13]:
linest_metrics = lda_metrics(linest_coefficients, X_design, y)

linest_train_predictions = linest_metrics["predictions"]
linest_cutoffs = linest_metrics["cutoffs"]
linest_predicted_classes = classify_by_cutoffs(linest_train_predictions, linest_cutoffs)

linest_difference = count_errors(train["Vendor"], linest_predicted_classes)

print("LINEST means:")
print(linest_metrics["means"])

print("\nLINEST sample numbers:")
print(linest_metrics["counts"])

print("\nLINEST cutoffs:")
print(linest_cutoffs)

print("\nLINEST Difference =", linest_difference)
print("Inter-group variance =", linest_metrics["inter_group"])
print("Within-group variance =", linest_metrics["within_group"])
print("Inter/within ratio =", linest_metrics["ratio"])

LINEST means:
y
0.0    0.098456
1.0    0.985982
2.0    1.899716
Name: prediction, dtype: float64

LINEST sample numbers:
y
0.0    59
1.0    71
2.0    48
Name: prediction, dtype: int64

LINEST cutoffs:
[np.float64(0.5831819188947683), np.float64(1.354546844431928)]

LINEST Difference = 13
Inter-group variance = 1.6223819580279275
Within-group variance = 9.553963755415872
Inter/within ratio = 0.16981244638993373


In [14]:
# Классификация неизвестных строк 182-184 через LINEST

X_unknown = unknown[features].astype(float).values
X_unknown_design = np.column_stack([np.ones(len(X_unknown)), X_unknown])

unknown_predictions_linest = calculate_predictions(X_unknown_design, linest_coefficients)
unknown_classes_linest = classify_by_cutoffs(unknown_predictions_linest, linest_cutoffs)

unknown_result_linest = unknown[["Vendor"] + features].copy()
unknown_result_linest["Numerical prediction"] = unknown_predictions_linest
unknown_result_linest["Predicted vendor"] = unknown_classes_linest

unknown_result_linest

,Vendor,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,Numerical prediction,Predicted vendor
180,?,14.0,4.0,3.0,19.0,97.0,3.0,1.0,1.0,2.0,5.0,1.0,3.0,910.0,0.836467,V2
181,?,13.0,1.0,2.0,14.0,105.0,3.0,1.0,1.0,1.0,7.0,1.0,2.0,1660.0,0.664412,V2
182,?,12.0,1.0,2.0,28.0,82.0,1.0,4.0,1.0,2.0,7.0,1.0,3.0,489.0,0.535162,V1


In [15]:
## 2. Solver-аналог

#Теперь оптимизируем коэффициенты так, как Solver в Excel: максимизируем отношение

#\[
#\frac{S_G}{S_W}
#\]

#где:

#- \(S_G\) — межгрупповая дисперсия;
#- \(S_W\) — внутригрупповая дисперсия.

#В Excel это делалось через Solver. В Python используем `scipy.optimize`.


In [16]:
from scipy.optimize import minimize

def objective(coefficients):
    metrics = lda_metrics(coefficients, X_design, y)
    return -metrics["ratio"]  # maximize ratio -> minimize negative ratio


# Стартуем с коэффициентов LINEST, как в Excel
start_coefficients = linest_coefficients.copy()

solver_result = minimize(
    objective,
    start_coefficients,
    method="BFGS",
    options={
        "maxiter": 10000,
        "gtol": 1e-9
    }
)

solver_coefficients = solver_result.x

solver_table = pd.DataFrame({
    "coefficient": coef_names,
    "LINEST": linest_coefficients,
    "Solver-like": solver_coefficients
})

solver_table

,coefficient,LINEST,Solver-like
0,b,3.473285,3.473285
1,w1,-0.117004,-0.078947
2,w2,0.030171,0.040162
3,w3,-0.148552,-0.053954
4,w4,0.039854,0.032516
5,w5,-0.000490,-0.000484
6,w6,0.144320,0.136342
7,w7,-0.372391,-0.373290
8,w8,-0.303474,-0.350144
9,w9,0.039357,0.026038


In [17]:
solver_metrics = lda_metrics(solver_coefficients, X_design, y)

solver_train_predictions = solver_metrics["predictions"]
solver_cutoffs = solver_metrics["cutoffs"]
solver_predicted_classes = classify_by_cutoffs(solver_train_predictions, solver_cutoffs)

solver_difference = count_errors(train["Vendor"], solver_predicted_classes)

print("Solver-like means:")
print(solver_metrics["means"])

print("\nSolver-like sample numbers:")
print(solver_metrics["counts"])

print("\nSolver-like cutoffs:")
print(solver_cutoffs)

print("\nSolver-like Difference =", solver_difference)
print("Inter-group variance =", solver_metrics["inter_group"])
print("Within-group variance =", solver_metrics["within_group"])
print("Inter/within ratio =", solver_metrics["ratio"])

Solver-like means:
y
0.0    0.866994
1.0    1.557312
2.0    2.579469
Name: prediction, dtype: float64

Solver-like sample numbers:
y
0.0    59
1.0    71
2.0    48
Name: prediction, dtype: int64

Solver-like cutoffs:
[np.float64(1.2440137134453668), np.float64(1.96961062940652)]

Solver-like Difference = 14
Inter-group variance = 1.4846387968948056
Within-group variance = 8.588330316932852
Inter/within ratio = 0.17286698835601075


In [18]:
# Классификация неизвестных строк 182-184 через Solver-like коэффициенты

unknown_predictions_solver = calculate_predictions(X_unknown_design, solver_coefficients)
unknown_classes_solver = classify_by_cutoffs(unknown_predictions_solver, solver_cutoffs)

unknown_result_solver = unknown[["Vendor"] + features].copy()
unknown_result_solver["Numerical prediction"] = unknown_predictions_solver
unknown_result_solver["Predicted vendor"] = unknown_classes_solver

unknown_result_solver


,Vendor,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,Numerical prediction,Predicted vendor
180,?,14.0,4.0,3.0,19.0,97.0,3.0,1.0,1.0,2.0,5.0,1.0,3.0,910.0,1.606629,V2
181,?,13.0,1.0,2.0,14.0,105.0,3.0,1.0,1.0,1.0,7.0,1.0,2.0,1660.0,1.423838,V2
182,?,12.0,1.0,2.0,28.0,82.0,1.0,4.0,1.0,2.0,7.0,1.0,3.0,489.0,1.004868,V1


In [19]:
## 3. Сравнение итогов

#Главный результат задания — классификация неизвестных вин в строках 182–184.


In [20]:
comparison = pd.DataFrame({
    "Row": unknown.index + 2,  # Excel rows
    "LINEST prediction": unknown_predictions_linest,
    "LINEST vendor": unknown_classes_linest,
    "Solver prediction": unknown_predictions_solver,
    "Solver vendor": unknown_classes_solver
})

comparison

,Row,LINEST prediction,LINEST vendor,Solver prediction,Solver vendor
0,182,0.836467,V2,1.606629,V2
1,183,0.664412,V2,1.423838,V2
2,184,0.535162,V1,1.004868,V1


In [21]:
# Сохраняем результат в Excel

output_file = "Chapter4_HW_LDA_Python_Result.xlsx"

with pd.ExcelWriter(output_file) as writer:
    linest_table.to_excel(writer, sheet_name="LINEST coefficients", index=False)
    solver_table.to_excel(writer, sheet_name="Solver coefficients", index=False)
    comparison.to_excel(writer, sheet_name="Unknown classification", index=False)

print(f"Результат сохранен в файл: {output_file}")

Результат сохранен в файл: Chapter4_HW_LDA_Python_Result.xlsx
